# Gợi ý vị trí thực tập IT và đánh giá mô hình

Notebook nhận `bo_du_lieu_thuc_tap_IT.csv` đã tổng hợp, hồ sơ sinh viên và nhãn cặp sinh viên–vị trí do người chấm cung cấp. Chạy phần gợi ý Top-K không cần train. Phần train/đánh giá chỉ chạy được khi có dữ liệu nhãn thật.

**Ý nghĩa nhãn:** 0 = không phù hợp, 1 = thấp, 2 = khá, 3 = phù hợp cao. Trong các code gửi kèm, nhãn được sinh bằng luật trùng kỹ năng/vị trí nên nếu dùng nhãn luật thì kết quả chỉ đo mức mô hình bắt chước luật, không chứng minh hiệu quả thực tế.

**Tỷ lệ chia:** 70% train, 15% validation, 15% test theo `ma_ho_so`, tránh hồ sơ của cùng sinh viên lọt vào nhiều tập. Cần gán nhãn thủ công/được xác nhận trước khi dùng số liệu làm kết quả chính thức.


In [ ]:
!pip -q install pandas scikit-learn


## 1. Nạp dữ liệu
Tải lên CSV danh sách việc làm đã gộp, CSV hồ sơ sinh viên (cột `ma_ho_so, nhom, ky_nang`, tùy chọn `dia_diem_mong_muon`) và CSV nhãn (cột `ma_ho_so, source_record_id, nhan`, nhãn 0–3). Dùng nút upload của Colab.


In [ ]:
from google.colab import files
import io, pandas as pd, numpy as np

up = files.upload()
def read_csv_by_hint(hint):
    names = [n for n in up if hint.lower() in n.lower()]
    if len(names) != 1:
        raise ValueError(f"Cần upload đúng một file chứa tên '{hint}'. Các file hiện có: {list(up)}")
    return pd.read_csv(io.BytesIO(up[names[0]]), encoding='utf-8-sig')

jobs = read_csv_by_hint('bo_du_lieu_thuc_tap_IT')
print('Số dòng việc làm:', len(jobs))
print('Các cột:', jobs.columns.tolist())
display(jobs[['source_dataset','source_record_id','job_title','company_name','location']].head())


## 2. Top-K gợi ý theo nội dung
Nhập kỹ năng, ngành/vị trí và địa điểm của sinh viên. Điểm dưới đây là **độ tương đồng TF-IDF**, không phải xác suất được nhận hay nhãn đã được kiểm chứng.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def txt(v):
    return '' if pd.isna(v) else str(v)
job_text = (jobs['job_title'].map(txt) + ' ' + jobs['skills_or_tags'].map(txt) + ' ' +
            jobs['job_description'].map(txt) + ' ' + jobs['company_expertise'].map(txt))
vectorizer = TfidfVectorizer(ngram_range=(1,2), min_df=1, max_features=50000, strip_accents='unicode')
J = vectorizer.fit_transform(job_text)

# Điền thông tin sinh viên ở đây
ky_nang_sv = 'Python, SQL, Java'
nganh_va_vi_tri = 'Công nghệ thông tin, Backend Developer'
dia_diem_sv = 'Ho Chi Minh'
profile_text = f'{ky_nang_sv} {nganh_va_vi_tri} {dia_diem_sv}'
q = vectorizer.transform([profile_text])
scores = cosine_similarity(q, J).ravel()
top_k = 10
ix = np.argsort(scores)[::-1][:top_k]
top = jobs.iloc[ix][['source_dataset','source_record_id','job_title','company_name','location','skills_or_tags']].copy()
top.insert(0, 'thu_hang', range(1, len(top)+1))
top['diem_tuong_dong_%'] = (scores[ix] * 100).round(2)
display(top)
top.to_csv('top_k_goi_y.csv', index=False, encoding='utf-8-sig')
files.download('top_k_goi_y.csv')


## 3. Huấn luyện và đánh giá (cần nhãn)
Upload thêm `mau_ho_so_sinh_vien.csv` có `ma_ho_so, nhom, ky_nang` và `mau_nhan_cap_sinh_vien_vi_tri.csv` có `ma_ho_so, ma_jd, nhan`. Chỉ gán nhãn cho vị trí đã được người chấm xem xét. Dòng nhãn trùng sẽ được gộp; mã vị trí phải trùng với `source_record_id` trong danh sách việc làm.


In [ ]:
profiles = read_csv_by_hint('mau_ho_so_sinh_vien')
labels = read_csv_by_hint('mau_nhan_cap_sinh_vien_vi_tri')
required_p = {'ma_ho_so','ky_nang'}
required_l = {'ma_ho_so','ma_jd','nhan'}
assert required_p <= set(profiles), f"Thiếu cột hồ sơ: {required_p-set(profiles)}"
assert required_l <= set(labels), f"Thiếu cột nhãn: {required_l-set(labels)}"
profiles['ma_ho_so'] = profiles['ma_ho_so'].astype(str)
labels['ma_ho_so'] = labels['ma_ho_so'].astype(str)
if 'ma_jd' not in jobs: jobs['ma_jd'] = [f"{src}::{rid}::row{i}" for i,(src,rid) in enumerate(zip(jobs['source_dataset'].fillna('').astype(str), jobs['source_record_id'].fillna('').astype(str)))]
labels['ma_jd'] = labels['ma_jd'].fillna('').astype(str)
labels['nhan'] = pd.to_numeric(labels['nhan'], errors='coerce')
labels = labels.dropna(subset=['nhan'])
labels = labels[labels['nhan'].isin([0,1,2,3])].copy()
labels['nhan'] = labels['nhan'].astype(int)

# Ghép theo mã vị trí duy nhất, không nhầm ID giữa các nguồn.
job_lookup = jobs.set_index('ma_jd')
labels = labels[labels['ma_jd'].isin(job_lookup.index)].copy()
prof_lookup = profiles.drop_duplicates('ma_ho_so').set_index('ma_ho_so')
labels = labels[labels['ma_ho_so'].isin(prof_lookup.index)].copy()
if labels.empty: raise ValueError('Không ghép được nhãn với hồ sơ/vị trí; kiểm tra mã ID.')

# Tạo đặc trưng cặp bằng độ tương đồng nội dung. Nhãn không tham gia tạo đặc trưng.
p_text = prof_lookup['ky_nang'].map(txt)
if 'nhom' in prof_lookup: p_text = p_text + ' ' + prof_lookup['nhom'].map(txt)
if 'dia_diem_mong_muon' in prof_lookup: p_text = p_text + ' ' + prof_lookup['dia_diem_mong_muon'].map(txt)
p_vec = vectorizer.transform(p_text.tolist())
pi = {m:i for i,m in enumerate(prof_lookup.index)}
ji = {m:i for i,m in enumerate(jobs['ma_jd'])}
X = np.array([cosine_similarity(p_vec[pi[m]], J[ji[j]])[0,0] for m,j in zip(labels.ma_ho_so, labels.ma_jd)]).reshape(-1,1)
y = labels['nhan'].to_numpy()
groups = labels['ma_ho_so'].to_numpy()
print('Số cặp có nhãn:', len(labels), '| hồ sơ:', labels.ma_ho_so.nunique(), '| phân bố nhãn:', pd.Series(y).value_counts().sort_index().to_dict())


In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, confusion_matrix, classification_report
from sklearn.metrics import ndcg_score

# Tách theo sinh viên: 70% train, 15% validation, 15% test (xấp xỉ theo số hồ sơ).
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
tr, hold = next(gss1.split(X, y, groups))
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.50, random_state=42)
va_rel, te_rel = next(gss2.split(X[hold], y[hold], groups[hold]))
va, te = hold[va_rel], hold[te_rel]
print('Cặp train/val/test:', len(tr), len(va), len(te))
print('Tỷ lệ cặp %:', *(round(100*n/len(y),1) for n in (len(tr),len(va),len(te))))
print('Hồ sơ tách biệt:', len(set(groups[tr]) & set(groups[va]))==0 and len(set(groups[tr]) & set(groups[te]))==0 and len(set(groups[va]) & set(groups[te]))==0)

model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=42, min_samples_leaf=2)
model.fit(X[tr], y[tr])
if len(va): print('Validation macro-F1:', round(f1_score(y[va], model.predict(X[va]), average='macro'),4))
pred = model.predict(X[te])
print('TEST accuracy:', round(accuracy_score(y[te], pred),4))
print('TEST macro-F1:', round(f1_score(y[te], pred, average='macro', zero_division=0),4))
print('TEST MAE (bậc nhãn 0–3):', round(mean_absolute_error(y[te], pred),4))
print('Ma trận nhầm lẫn (nhãn 0,1,2,3):\n', confusion_matrix(y[te], pred, labels=[0,1,2,3]))
print(classification_report(y[te], pred, labels=[0,1,2,3], zero_division=0))


In [ ]:
# NDCG@5/10 trên các cặp test có nhãn; nhãn 2–3 được coi là phù hợp.
test_rows = labels.iloc[te].copy()
test_rows['score_pred'] = model.predict_proba(X[te]) @ model.classes_
test_rows['relevant'] = (test_rows['nhan'] >= 2).astype(int)
def ndcg_group(g, k):
    if g['relevant'].sum() == 0: return np.nan
    order = np.argsort(g['score_pred'].to_numpy())[::-1][:k]
    rel = g['relevant'].to_numpy()[order]
    dcg = np.sum(rel / np.log2(np.arange(2, len(rel)+2)))
    ideal = np.sort(g['relevant'].to_numpy())[::-1][:k]
    idcg = np.sum(ideal / np.log2(np.arange(2, len(ideal)+2)))
    return dcg/idcg if idcg else np.nan
for k in (5,10):
    vals = test_rows.groupby('ma_ho_so').apply(lambda g: ndcg_group(g,k), include_groups=False).dropna()
    print(f'NDCG@{k} test =', round(vals.mean(),4) if len(vals) else 'không đủ nhãn liên quan')
test_rows['du_doan_nhan'] = pred
test_rows.to_csv('ket_qua_test.csv', index=False, encoding='utf-8-sig')
files.download('ket_qua_test.csv')


## Diễn giải kết quả
- **Accuracy / macro-F1 / MAE** đo dự đoán nhãn phù hợp 0–3 trên các cặp test đã gán nhãn.
- **NDCG@5/10** đo chất lượng thứ tự Top-K trên các ứng viên đã được gán nhãn; chỉ đáng tin khi nhãn test do người đánh giá xác nhận và tập test đủ lớn.
- Điểm TF-IDF ở phần 2 là độ tương đồng văn bản, không phải xác suất.
- Nếu dùng `nhan` được tạo tự động bằng luật trong `external_dataset.py`, hãy ghi rõ đây là đánh giá trên nhãn yếu/tự sinh. Không báo cáo số đó như độ chính xác thực tế của tư vấn thực tập.
- Nếu mẫu nhỏ hoặc một lớp vắng mặt trong test, macro-F1 và NDCG có thể không ổn định; cần thêm cặp đã gán nhãn.
